# Week 14 Walkthrough — A Robot That Finds Its Way

**Topic:** Autonomous behaviour, decision-making, logging, project structure

Last week's robot turned when it bumped into things. That's reflex, not intelligence. This week it gets a goal, a strategy for reaching it, a log of why it did what it did, and a summary at the end — the shape your capstone submission should take.

Run the cells in order. Each step adds one idea to the program, and the last
section pulls the whole thing together. Change things and re-run — that is the
whole point of a notebook.

## Step 1 — Start from last week's world


Same GridWorld, same idea. We'll build the smarter robot on top.

In [ ]:
GRID = [
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0, 0, 0],
]

class GridWorld:
    def __init__(self, grid, goal):
        self.grid = grid
        self.rows, self.cols = len(grid), len(grid[0])
        self.goal = goal

    def is_open(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols and self.grid[r][c] == 0


world = GridWorld(GRID, goal=(6, 6))
print(f"{world.rows}x{world.cols} world, goal at {world.goal}")

## Step 2 — Reflex is not enough


A robot that only turns when blocked can loop forever. Watch it fail — knowing
*why* a naive strategy fails is the point of this step.

In [ ]:
HEADINGS = ["north", "east", "south", "west"]
MOVES = {"north": (-1, 0), "east": (0, 1), "south": (1, 0), "west": (0, -1)}

row, col, heading = 0, 0, "east"
visited = []

for tick in range(20):
    visited.append((row, col))
    dr, dc = MOVES[heading]
    if world.is_open(row + dr, col + dc):
        row, col = row + dr, col + dc
    else:
        heading = HEADINGS[(HEADINGS.index(heading) + 1) % 4]

print("ended at", (row, col), "goal is", world.goal)
print("distinct squares visited:", len(set(visited)), "of", 20)

## Step 3 — Give it a preference: move toward the goal


Score each option by how much closer it gets us. This is the seed of every
path-finding algorithm.

In [ ]:
def distance(a, b):
    """Manhattan distance - steps on a grid, no diagonals."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


here = (0, 0)
print("distance to goal:", distance(here, world.goal))

for heading, (dr, dc) in MOVES.items():
    nxt = (here[0] + dr, here[1] + dc)
    ok = world.is_open(*nxt)
    print(f"{heading:6} -> {nxt}  open={ok}  distance={distance(nxt, world.goal)}")

## Step 4 — Choose the best legal move


Filter to what's possible, then pick the option that scores best. Prefer keeping
the current heading on ties so it doesn't dither.

In [ ]:
def best_move(position, heading, world):
    """Return the heading that gets us closest, or None if boxed in."""
    options = []
    for h, (dr, dc) in MOVES.items():
        nxt = (position[0] + dr, position[1] + dc)
        if world.is_open(*nxt):
            turn_cost = 0 if h == heading else 1
            options.append((distance(nxt, world.goal), turn_cost, h))
    if not options:
        return None
    options.sort()
    return options[0][2]


print(best_move((0, 0), "east", world))
print(best_move((2, 4), "north", world))

## Step 5 — Remember where you've been


A greedy robot walks into dead ends and oscillates. A `visited` set breaks the
tie and stops the loop.

In [ ]:
def best_move(position, heading, world, visited):
    options = []
    for h, (dr, dc) in MOVES.items():
        nxt = (position[0] + dr, position[1] + dc)
        if not world.is_open(*nxt):
            continue
        seen_penalty = 1 if nxt in visited else 0
        turn_cost = 0 if h == heading else 1
        options.append((seen_penalty, distance(nxt, world.goal), turn_cost, h))
    if not options:
        return None
    options.sort()
    return options[0][3]


print(best_move((0, 0), "east", world, visited={(0, 1)}))

## Step 6 — Log the reasoning, not just the moves


"Moved east" tells you what happened. "Moved east — 9 squares from goal, 2 "
"options considered" tells you *why*. Your reflection paper will thank you.

In [ ]:
log = []
log.append({"tick": 1, "at": (0, 0), "chose": "east",
            "distance": 12, "options": 2, "why": "closest legal square"})

for entry in log:
    print(f"t{entry['tick']:>3}  {entry['at']}  -> {entry['chose']:5} "
          f"(d={entry['distance']}, {entry['options']} options) {entry['why']}")

---

## The finished program

Everything above, in one place. This is the version worth keeping.


A complete autonomous run: sense, think with a strategy, act, log, and report.
This is roughly the scope of a solid capstone — yours should add your own project
theme on top.

In [ ]:
# Week 14 - An autonomous robot with a goal

class GridWorld:
    def __init__(self, grid, goal):
        self.grid = grid
        self.rows, self.cols = len(grid), len(grid[0])
        self.goal = goal

    def is_open(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols and self.grid[r][c] == 0


class Navigator:
    """A robot that steers toward a goal and explains itself."""

    HEADINGS = ["north", "east", "south", "west"]
    MOVES = {"north": (-1, 0), "east": (0, 1), "south": (1, 0), "west": (0, -1)}
    ARROWS = {"north": "^", "east": ">", "south": "v", "west": "<"}

    def __init__(self, world, start=(0, 0), heading="east", battery=60):
        self.world = world
        self.row, self.col = start
        self.heading = heading
        self.battery = battery
        self.visited = {start}
        self.log = []
        self.ticks = 0

    # --- sensing -------------------------------------------------
    @property
    def position(self):
        return (self.row, self.col)

    def distance_to_goal(self, position=None):
        r, c = position or self.position
        gr, gc = self.world.goal
        return abs(r - gr) + abs(c - gc)

    def at_goal(self):
        return self.position == self.world.goal

    def options(self):
        """Every legal move from here, scored."""
        found = []
        for h, (dr, dc) in self.MOVES.items():
            nxt = (self.row + dr, self.col + dc)
            if not self.world.is_open(*nxt):
                continue
            found.append((
                1 if nxt in self.visited else 0,     # prefer new ground
                self.distance_to_goal(nxt),          # then prefer closer
                0 if h == self.heading else 1,       # then avoid turning
                h, nxt,
            ))
        found.sort()
        return found

    # --- thinking + acting ---------------------------------------
    def step(self):
        self.ticks += 1
        self.battery -= 1

        choices = self.options()
        if not choices:
            self.log.append((self.ticks, self.position, None, "boxed in"))
            return False

        _, dist, _, heading, nxt = choices[0]
        why = "new ground" if nxt not in self.visited else "backtracking"

        self.heading = heading
        self.row, self.col = nxt
        self.visited.add(nxt)
        self.log.append((self.ticks, nxt, heading, f"{why}, d={dist}"))
        return True

    def run(self, max_ticks=80):
        while not self.at_goal() and self.battery > 0 and self.ticks < max_ticks:
            if not self.step():
                break
        return self.at_goal()

    # --- reporting -----------------------------------------------
    def render(self):
        out = []
        for r in range(self.world.rows):
            row = []
            for c in range(self.world.cols):
                if (r, c) == self.position:
                    row.append(self.ARROWS[self.heading])
                elif (r, c) == self.world.goal:
                    row.append("G")
                elif self.world.grid[r][c]:
                    row.append("#")
                elif (r, c) in self.visited:
                    row.append("+")
                else:
                    row.append(".")
            out.append(" ".join(row))
        return "\n".join(out)

    def summary(self):
        reached = "REACHED GOAL" if self.at_goal() else "did not reach goal"
        return (
            f"{reached} in {self.ticks} ticks\n"
            f"battery left:     {self.battery}\n"
            f"squares visited:  {len(self.visited)}\n"
            f"final position:   {self.position} facing {self.heading}"
        )


GRID = [
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0, 0, 0],
]

world = GridWorld(GRID, goal=(6, 6))
robot = Navigator(world)

print("START")
print(robot.render())

robot.run()

print("\nEND  (+ marks squares visited)")
print(robot.render())
print()
print(robot.summary())

print("\nDECISION LOG (first 12)")
for tick, pos, heading, why in robot.log[:12]:
    print(f"  t{tick:>3}  {str(pos):8} {str(heading):6} {why}")

---

## Try it yourself

Use the empty cells below. There is no grade attached — this is where the
learning actually happens.

**1.** Move the goal to a corner behind a wall. Does it still get there? If not, what would it need to remember?

**2.** Drop `battery` to 15 and re-run. Does the summary report the failure honestly?

**3.** Add a `sensor_range` so the robot only sees one square ahead — a much harder and more realistic problem.

**4.** Swap the scoring in `options()` so it prefers turning as little as possible above all else. Watch the path change shape.

**5.** This is your capstone skeleton. Rename `Navigator` to match your chosen project and give it behaviour specific to that theme.

In [ ]:
# Try it yourself 1

In [ ]:
# Try it yourself 2

In [ ]:
# Try it yourself 3

In [ ]:
# Try it yourself 4